# MIMIC-III Experiment Analysis

Analysis of VCIP and baseline model results on the MIMIC-III real ICU dataset.

**Dataset:** MIMIC-III v1.4 (hourly-averaged ICU data)  
**Treatments:** vasopressor, mechanical ventilation (binary)  
**Outcome:** diastolic blood pressure  
**Models:** VCIP, ACTIN, CT, CRN, RMSN  
**Metrics:** GRP (Goal-Reaching Probability), RCS (Rank Correlation with Surrogate), MSE

In [ ]:
import pickle
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add parent results directory to path for reusing tools.py
sys.path.insert(0, str(Path('../all').resolve()))
from tools import plot_combined_distributions, plot_rank_distributions_combined

## 1. Load Case Info Data (GRP / RCS)

In [ ]:
def load_mimic_case_infos(base_dir):
    """
    Load case_infos data from MIMIC-III output directory.
    
    MIMIC outputs have a different structure than Cancer:
    my_outputs/mimic_real/{model_name}/train/case_infos/{seed}/{test}/case_infos_{model}.pkl
    
    Returns: {model_name: {tau: [case_info_dicts]}}
    """
    base_path = Path(base_dir)
    all_data = {}
    
    for model_dir in base_path.iterdir():
        if not model_dir.is_dir():
            continue
        
        model_name = model_dir.name
        all_data[model_name] = {}
        
        # Search for case_infos directories
        for case_infos_dir in model_dir.rglob('case_infos'):
            if not case_infos_dir.is_dir():
                continue
            
            for seed_dir in case_infos_dir.iterdir():
                if not seed_dir.is_dir():
                    continue
                
                # Look in both True and False subdirectories
                for test_dir in seed_dir.iterdir():
                    if not test_dir.is_dir():
                        continue
                    
                    for pkl_file in test_dir.glob('case_infos_*.pkl'):
                        with open(pkl_file, 'rb') as f:
                            data = pickle.load(f)
                        
                        for tau, cases in data.items():
                            if tau not in all_data[model_name]:
                                all_data[model_name][tau] = []
                            all_data[model_name][tau].extend(cases)
    
    return all_data


mimic_output_dir = '../../my_outputs/mimic_real'
case_data = load_mimic_case_infos(mimic_output_dir)
print('Models found:', list(case_data.keys()))
for model, taus in case_data.items():
    print(f'  {model}: tau values = {sorted(taus.keys())}, total cases per tau = {[len(taus[t]) for t in sorted(taus.keys())]}')

## 2. GRP and RCS Distributions

In [ ]:
# Plot GRP and RCS using the shared plotting function
tau_values = [2, 4, 6]
fig = plot_combined_distributions(case_data, tau_values=tau_values, vi=False)
plt.savefig('mimic_grp_rcs.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 3. GRP/RCS Summary Table (mean +/- std across seeds)

In [ ]:
models_order = ['VCIP', 'ACTIN', 'CT', 'CRN', 'RMSN']

def compute_grp_rcs_summary(case_data, tau_values):
    """Compute mean and std of GRP and RCS for each model and tau."""
    rows = []
    for model in models_order:
        if model not in case_data:
            continue
        for tau in tau_values:
            if tau not in case_data[model]:
                continue
            cases = case_data[model][tau]
            
            # GRP
            ranks = np.array([c['true_sequence_rank'] for c in cases])
            n = 100
            grp = (n - ranks) / (n - 1)
            
            # RCS
            corrs = np.array([c['correlations']['model_true'] for c in cases])
            corrs = corrs[~np.isnan(corrs)]
            
            rows.append({
                'Model': model,
                'tau': tau,
                'GRP_mean': np.mean(grp),
                'GRP_std': np.std(grp),
                'RCS_mean': np.mean(corrs),
                'RCS_std': np.std(corrs),
            })
    
    return pd.DataFrame(rows)

summary = compute_grp_rcs_summary(case_data, tau_values=[2, 4, 6])
print(summary.to_string(index=False))

## 4. MSE Over Tau (One-Step-Ahead Prediction)

In [ ]:
def load_mimic_mse(base_dir):
    """Load mse.csv files from MIMIC output directory."""
    base_path = Path(base_dir)
    mse_data = {}
    
    for model_dir in base_path.iterdir():
        if not model_dir.is_dir():
            continue
        
        model_name = model_dir.name
        
        for mse_file in model_dir.rglob('mse.csv'):
            df = pd.read_csv(mse_file, index_col=False)
            if 'seed' in df.columns:
                df = df.drop('seed', axis=1)
            mse_data[model_name] = df
    
    return mse_data

mse_data = load_mimic_mse(mimic_output_dir)
print('Models with MSE data:', list(mse_data.keys()))

In [ ]:
# Plot MSE over tau for all models
max_tau = 6
colors = sns.color_palette('Set2', len(models_order))
markers = ['o', 's', '^', '<', '*']

fig, ax = plt.subplots(1, 1, figsize=(8, 5))

for i, model in enumerate(models_order):
    if model not in mse_data:
        continue
    df = mse_data[model]
    mse_values = df.mean()
    mse_std = df.std()
    taus = range(1, min(max_tau + 1, len(mse_values) + 1))
    
    ax.errorbar(taus, mse_values[:max_tau], yerr=mse_std[:max_tau],
                label=model, color=colors[i], marker=markers[i],
                linewidth=1.5, markersize=6, markerfacecolor='white',
                markeredgewidth=2, capsize=3)

ax.set_xlabel(r'$\tau$', fontsize=14)
ax.set_ylabel('Mean MSE', fontsize=14)
ax.set_title('MIMIC-III: Multi-step Prediction MSE', fontsize=14)
ax.set_xticks(range(1, max_tau + 1))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mimic_mse_trends.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 5. Slice Discovery: Where Does VCIP Fail?

Segment patients by:
- Static features: age quartile, gender, ethnicity
- Treatment pattern: vasopressor-only, ventilation-only, both, neither

In [ ]:
def load_mimic_dataset_for_slicing(data_path):
    """
    Load the MIMIC-III dataset to get patient-level static features
    and treatment patterns for slice analysis.
    """
    sys.path.insert(0, str(Path('../../src').resolve()))
    from data.mimic_iii.load_data import load_mimic3_data_processed
    
    treatments, outcomes, vitals, static_features, outcomes_unscaled, scaling_params = \
        load_mimic3_data_processed(
            data_path,
            min_seq_length=60,
            max_seq_length=60,
            max_number=1000
        )
    
    return treatments, static_features, outcomes_unscaled

# Uncomment when data is available:
# data_path = '../../data/processed/all_hourly_data.h5'
# treatments, static_features, outcomes_unscaled = load_mimic_dataset_for_slicing(data_path)

In [ ]:
def analyze_slices(case_data, treatments, static_features, tau=4):
    """
    Analyze GRP performance by patient slices.
    
    Args:
        case_data: dict {model: {tau: [case_dicts]}}
        treatments: DataFrame with treatment columns (vaso, vent)
        static_features: DataFrame with patient static features
        tau: prediction horizon to analyze
    """
    # Get unique patient IDs from treatments
    patient_ids = treatments.index.get_level_values('subject_id').unique()
    
    # Treatment pattern per patient
    patient_treatments = treatments.groupby('subject_id').max()
    patient_treatments['pattern'] = 'neither'
    mask_vaso = patient_treatments['vaso'] > 0
    mask_vent = patient_treatments['vent'] > 0
    patient_treatments.loc[mask_vaso & ~mask_vent, 'pattern'] = 'vaso_only'
    patient_treatments.loc[~mask_vaso & mask_vent, 'pattern'] = 'vent_only'
    patient_treatments.loc[mask_vaso & mask_vent, 'pattern'] = 'both'
    
    print('Treatment pattern distribution:')
    print(patient_treatments['pattern'].value_counts())
    print()
    
    # Per-model GRP by treatment pattern
    models_order = ['VCIP', 'ACTIN', 'CT', 'CRN', 'RMSN']
    for model in models_order:
        if model not in case_data or tau not in case_data[model]:
            continue
        cases = case_data[model][tau]
        ranks = np.array([c['true_sequence_rank'] for c in cases])
        grp = (100 - ranks) / 99
        print(f'{model} (tau={tau}): GRP = {np.mean(grp):.3f} +/- {np.std(grp):.3f}')
    
    return patient_treatments

# Uncomment when data and results are available:
# patient_treatments = analyze_slices(case_data, treatments, static_features, tau=4)

In [ ]:
def compare_vcip_vs_baselines(case_data, tau=4):
    """
    Identify patients where VCIP underperforms the best baseline.
    Returns indices of failure cases for further investigation.
    """
    if 'VCIP' not in case_data or tau not in case_data['VCIP']:
        print('VCIP data not available')
        return None
    
    vcip_cases = case_data['VCIP'][tau]
    vcip_grp = np.array([(100 - c['true_sequence_rank']) / 99 for c in vcip_cases])
    
    # Find best baseline GRP per patient
    baselines = ['ACTIN', 'CT', 'CRN', 'RMSN']
    best_baseline_grp = np.zeros_like(vcip_grp)
    best_baseline_name = [''] * len(vcip_grp)
    
    for bl in baselines:
        if bl not in case_data or tau not in case_data[bl]:
            continue
        bl_cases = case_data[bl][tau]
        if len(bl_cases) != len(vcip_cases):
            continue
        bl_grp = np.array([(100 - c['true_sequence_rank']) / 99 for c in bl_cases])
        better = bl_grp > best_baseline_grp
        best_baseline_grp[better] = bl_grp[better]
        for i in range(len(better)):
            if better[i]:
                best_baseline_name[i] = bl
    
    # Cases where VCIP is worse than best baseline
    failure_mask = vcip_grp < best_baseline_grp
    n_failures = failure_mask.sum()
    
    print(f'tau={tau}: VCIP underperforms best baseline in {n_failures}/{len(vcip_grp)} cases ({100*n_failures/len(vcip_grp):.1f}%)')
    print(f'  VCIP mean GRP: {vcip_grp.mean():.3f}')
    print(f'  Best baseline mean GRP: {best_baseline_grp.mean():.3f}')
    
    if n_failures > 0:
        print(f'  In failure cases — VCIP GRP: {vcip_grp[failure_mask].mean():.3f}, Best baseline GRP: {best_baseline_grp[failure_mask].mean():.3f}')
        # Count which baseline wins most often
        from collections import Counter
        winners = Counter(name for i, name in enumerate(best_baseline_name) if failure_mask[i])
        print(f'  Winning baselines: {dict(winners)}')
    
    return failure_mask

# Uncomment when results are available:
# failure_mask = compare_vcip_vs_baselines(case_data, tau=4)

## 6. Summary

Key findings from the MIMIC-III experiments:

1. **GRP/RCS performance:** (fill after experiments run)
2. **MSE trends:** (fill after experiments run)
3. **Failure modes:** (fill after slice discovery)
4. **Treatment pattern sensitivity:** (fill after analysis)